In [1]:
from pathlib import Path
import os
import json
import re
import gc

import pandas as pd
import torch
from PIL import Image
from tqdm.auto import tqdm

from transformers import AutoProcessor, AutoModelForImageTextToText
from peft import PeftModel

In [2]:
DATA_DIR = Path("/kaggle/input/competitions/pixels-to-predictions")

BASE_MODEL_DIR = "/kaggle/input/models/mutekik/base/other/default/1"
DORA_ADAPTER_DIR = "/kaggle/input/models/mutekik/r8-ep3-dor/other/default/1"

print("DATA_DIR exists:", DATA_DIR.exists())
print("BASE_MODEL_DIR exists:", os.path.exists(BASE_MODEL_DIR))
print("DORA_ADAPTER_DIR exists:", os.path.exists(DORA_ADAPTER_DIR))

print("\nBase model files:")
print(os.listdir(BASE_MODEL_DIR))

print("\nAdapter files:")
print(os.listdir(DORA_ADAPTER_DIR))

DATA_DIR exists: True
BASE_MODEL_DIR exists: True
DORA_ADAPTER_DIR exists: True

Base model files:
['config.json', 'merges.txt', 'preprocessor_config.json', 'README.md', 'tokenizer.json', 'vocab.json', 'tokenizer_config.json', 'chat_template.json', 'gitattributes', 'model.safetensors', 'processor_config.json', 'special_tokens_map.json', 'added_tokens.json', 'generation_config.json']

Adapter files:
['adapter_model.safetensors', 'adapter_config.json', 'README.md', 'tokenizer.json', 'tokenizer_config.json', 'chat_template.jinja', 'processor_config.json']


In [3]:
train_df = pd.read_csv(DATA_DIR / "train.csv")
val_df = pd.read_csv(DATA_DIR / "val.csv")
test_df = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

for df in [train_df, val_df, test_df]:
    df["choices"] = df["choices"].apply(json.loads)

print("train:", train_df.shape)
print("val:", val_df.shape)
print("test:", test_df.shape)
print("sample submission:", sample_submission.shape)

display(test_df.head())

train: (3109, 15)
val: (1048, 15)
test: (1008, 13)
sample submission: (1008, 2)


,id,image_path,question,choices,num_choices,hint,lecture,task,grade,subject,topic,category,skill
0,test_01750,images/test/test_01750.png,"Based on clues in the text, why would farmers ...",[The cats were thought to be visiting goddesse...,4,Read the text about cats.\nCats are among the ...,"Informational texts include many facts, exampl...",closed choice,grade5,language science,reading-comprehension,Informational texts: level 1,Read passages about animals
1,test_00128,images/test/test_00128.png,What is the probability that an American curl ...,"[0/4, 2/4, 4/4, 3/4, 1/4]",5,"In a group of American curl cats, some individ...",Offspring genotypes: homozygous or heterozygou...,closed choice,grade8,natural science,biology,Genes to traits,Use Punnett squares to calculate probabilities...
2,test_02891,images/test/test_02891.png,What is the expected ratio of offspring with a...,"[4:0, 3:1, 1:3, 0:4, 2:2]",5,This passage describes the ground spot color t...,Offspring phenotypes: dominant or recessive?\n...,closed choice,grade8,natural science,biology,Genes to traits,Use Punnett squares to calculate ratios of off...
3,test_02425,images/test/test_02425.png,What is the probability that a budgerigar para...,"[4/4, 2/4, 3/4, 1/4, 0/4]",5,"In a group of budgerigar parakeets, some indiv...",Offspring genotypes: homozygous or heterozygou...,closed choice,grade8,natural science,biology,Genes to traits,Use Punnett squares to calculate probabilities...
4,test_00930,images/test/test_00930.png,What is the probability that a muskmelon plant...,"[2/4, 1/4, 0/4, 3/4, 4/4]",5,"In a group of muskmelon plants, some individua...",Offspring genotypes: homozygous or heterozygou...,closed choice,grade8,natural science,biology,Genes to traits,Use Punnett squares to calculate probabilities...


In [4]:
def get_image_path(row):
    rel_path = Path(row["image_path"])


    p = DATA_DIR / "images" / rel_path
    if p.exists():
        return p

    p = DATA_DIR / rel_path
    if p.exists():
        return p

    matches = list(DATA_DIR.rglob(rel_path.name))
    if len(matches) > 0:
        return matches[0]

    raise FileNotFoundError(f"Could not find image: {row['image_path']}")

In [5]:
row = test_df.iloc[0]
img_path = get_image_path(row)

print(img_path)
print(img_path.exists())

img = Image.open(img_path).convert("RGB")
print(img.size)

/kaggle/input/competitions/pixels-to-predictions/images/images/test/test_01750.png
True
(302, 202)


In [6]:
LETTERS = "ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def clean_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

def index_to_letter(idx):
    return LETTERS[int(idx)]

def letter_to_index(letter):
    return LETTERS.index(letter.upper())

def build_prompt(row, include_answer=False):
    choices = row["choices"]

    choice_lines = []
    for i, choice in enumerate(choices):
        choice_lines.append(f"{LETTERS[i]}. {choice}")
    choices_text = "\n".join(choice_lines)

    parts = []

    for col in ["grade", "subject", "topic", "category", "skill"]:
        if col in row and clean_text(row[col]):
            parts.append(f"{col.capitalize()}: {clean_text(row[col])}")

    if "lecture" in row and clean_text(row["lecture"]):
        parts.append(f"Lecture:\n{clean_text(row['lecture'])}")

    if "hint" in row and clean_text(row["hint"]):
        parts.append(f"Hint:\n{clean_text(row['hint'])}")

    parts.append(f"Question:\n{clean_text(row['question'])}")
    parts.append(f"Choices:\n{choices_text}")

    valid_letters = ", ".join(LETTERS[:int(row["num_choices"])])
    parts.append(f"Answer with only one letter: {valid_letters}.")
    parts.append("Answer:")

    prompt = "\n\n".join(parts)

    if include_answer and "answer" in row:
        prompt += f" {index_to_letter(row['answer'])}"

    return prompt

In [7]:
print(build_prompt(test_df.iloc[0], include_answer=False))

Grade: grade5

Subject: language science

Topic: reading-comprehension

Category: Informational texts: level 1

Skill: Read passages about animals

Lecture:
Informational texts include many facts, examples, and details. Authors don't always directly state how these things connect to each other. So, you may need to make guesses, or inferences, to understand how the ideas from the text fit together. Inferences can help you understand the whole text and draw conclusions about the information. Be sure to base your inferences on details found in the text as well as things you already know.

Hint:
Read the text about cats.
Cats are among the most popular pets in the world. Millions of people have welcomed cats into their homes. Indeed, researchers believe that the relationship between cats and humans goes back to prehistoric times. But throughout history, different cultures and people around the world have had different sentiments about cats. Such feelings have ranged from fear to worship.
P

In [8]:
gc.collect()
torch.cuda.empty_cache()

processor = AutoProcessor.from_pretrained(
    BASE_MODEL_DIR,
    local_files_only=True
)

base_model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_DIR,
    dtype=torch.float16,
    device_map="auto",
    local_files_only=True
)

model = PeftModel.from_pretrained(
    base_model,
    DORA_ADAPTER_DIR,
    local_files_only=True
)

model.eval()

print("Loaded base model:", BASE_MODEL_DIR)
print("Loaded DoRA adapter:", DORA_ADAPTER_DIR)
print(type(model))
print("model device:", next(model.parameters()).device)

Loading weights:   0%|          | 0/489 [00:00<?, ?it/s]

Loaded base model: /kaggle/input/models/mutekik/base/other/default/1
Loaded DoRA adapter: /kaggle/input/models/mutekik/r8-ep3-dor/other/default/1
<class 'peft.peft_model.PeftModelForCausalLM'>
model device: cuda:0


In [9]:
model.print_trainable_parameters()

trainable params: 0 || all params: 512,100,544 || trainable%: 0.0000


In [10]:
def extract_answer_index(text, num_choices):
    text = str(text).strip().upper()
    valid_letters = LETTERS[:int(num_choices)]

    m = re.search(r"ANSWER\s*[:\-]?\s*([A-Z])", text)
    if m:
        letter = m.group(1)
        if letter in valid_letters:
            return letter_to_index(letter)

    m = re.search(r"\b([A-Z])\b", text)
    if m:
        letter = m.group(1)
        if letter in valid_letters:
            return letter_to_index(letter)

    nums = re.findall(r"\d+", text)
    for n in nums:
        ans = int(n)
        if 0 <= ans < int(num_choices):
            return ans

    return 0


def predict_one(row, max_new_tokens=5):
    prompt = build_prompt(row, include_answer=False)
    image = Image.open(get_image_path(row)).convert("RGB")

    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": prompt},
            ],
        }
    ]

    text = processor.apply_chat_template(
        messages,
        add_generation_prompt=True,
    )

    inputs = processor(
        text=text,
        images=image,
        return_tensors="pt",
    ).to(model.device)

    with torch.no_grad():
        generated_ids = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
        )

    new_tokens = generated_ids[:, inputs["input_ids"].shape[1]:]

    generated_text = processor.batch_decode(
        new_tokens,
        skip_special_tokens=True,
    )[0]

    pred = extract_answer_index(generated_text, row["num_choices"])
    return pred, generated_text

In [11]:
for i in range(3):
    row = val_df.iloc[i]
    pred, raw = predict_one(row, max_new_tokens=5)

    print("id:", row["id"])
    print("raw:", raw)
    print("pred:", pred)
    print("true:", int(row["answer"]))
    print()

id: val_00671
raw:  B
pred: 1
true: 0

id: val_04111
raw:  A
pred: 0
true: 1

id: val_02022
raw:  A
pred: 0
true: 3



In [12]:
submission_records = []

model.eval()

for i in tqdm(range(len(test_df))):
    row = test_df.iloc[i]
    pred, raw = predict_one(row, max_new_tokens=5)

    submission_records.append({
        "id": row["id"],
        "answer": int(pred)
    })

submission = pd.DataFrame(submission_records)

submission = sample_submission[["id"]].merge(
    submission, on="id", how="left"
)

submission["answer"] = submission["answer"].fillna(0).astype(int)

submission.to_csv("/kaggle/working/submission.csv", index=False)

display(submission.head())
print(submission.shape)
print(submission["answer"].value_counts().sort_index())
print("Saved to /kaggle/working/submission.csv")

  0%|          | 0/1008 [00:00<?, ?it/s]

,id,answer
0,test_01750,2
1,test_00128,0
2,test_02891,0
3,test_02425,3
4,test_00930,2


(1008, 2)
answer
0    370
1    340
2    224
3     74
Name: count, dtype: int64
Saved to /kaggle/working/submission.csv
